In [1]:
import numpy as np

def predict(x):
    return np.where(np.asarray(x)>=3,1,-1)

print(predict([2.5,3.5]))

[-1  1]


In [8]:
class LinearSVM:

    def __init__(self,learning_rate=0.01,epochs=100,C=0.01):
        self.learning_rate=learning_rate
        self.epochs=epochs
        self.C=C
        self.w=None
        self.b=None
        self.loss_history=[]

    def fit(self,x,y):
        x=np.asarray(x,dtype=float)
        y=np.asarray(y,dtype=float)

        classes=np.unique(y)
        if len(classes)!=2:
            raise ValueError('only one class , data is saturated')
        
        if not np.all(np.isin(classes,[-1,1])):
            raise ValueError('ENCODE LABELS IN -1 AMD +1')

        n_samples,n,features=x.shape
        self.w=np.zeroes(n_features)
        self.b=0.0

        for epoch in range(self.epochs):
            scores= x @ self.w + self.b

            margins=y*scores
            active=margins<1

            dw=self.w - self.C*(x[active].T @ y[active])/n_samples
            db=-self.C*np.sum(y[active])/n_samples

            self.w=self.w-self.learning_rate*dw
            self.b=self.b-self.learning_rate*db

            scores=x @ self.w + self.b
            hinge=np.maximum(0,1-y*scores)
            loss=0.5 * np.dot(self.w,self.w)+(self.C*np.mean(hinge))
            self.loss_history.append(loss)

        return self

    
    def decision_function(self,x):
        x=np.asarray(x,dtype=flaot)
        return x @ self.w +self.b

    def predict(self,x):
        scores=self.decision_function(x)
        return np.where(scores >=0,1,-1)


In [ ]:
    # -----------------------------------
# Test the scratch SVM on toy data
# -----------------------------------
if __name__ == "__main__":
    rng = np.random.default_rng(42)

    # Two separable groups
    X_negative = rng.normal(
        loc=[-2, -2], scale=0.8, size=(100, 2)
    )
    X_positive = rng.normal(
        loc=[2, 2], scale=0.8, size=(100, 2)
    )

    X = np.vstack([X_negative, X_positive])
    y = np.array([-1] * 100 + [1] * 100)

    # Shuffle observations
    indices = rng.permutation(len(y))
    X, y = X[indices], y[indices]

    # Standardize features using NumPy
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    X = (X - mean) / std

    # Train
    model = LinearSVM(
        learning_rate=0.01,
        C=1.0,
        epochs=2000
    )
    model.fit(X, y)

    # Evaluate
    predictions = model.predict(X)
    accuracy = np.mean(predictions == y)

    print("Weights:", model.w)
    print("Bias:", model.b)
    print(f"Training accuracy: {accuracy:.3f}")
    print(f"Final objective: {model.loss_history[-1]:.4f}")


In [10]:

import numpy as np


class LinearSVM:
    def __init__(self, learning_rate=0.001, C=1.0, epochs=2000):
        self.learning_rate = learning_rate
        self.C = C
        self.epochs = epochs
        self.w = None
        self.b = None
        self.loss_history = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        # SVM expects labels -1 and +1
        classes = np.unique(y)
        if len(classes) != 2:
            raise ValueError("This implementation supports binary classification.")

        if not np.all(np.isin(classes, [-1, 1])):
            raise ValueError("Encode labels as -1 and +1.")

        n_samples, n_features = X.shape

        self.w = np.zeros(n_features)
        self.b = 0.0

        for epoch in range(self.epochs):
            # Decision scores
            scores = X @ self.w + self.b

            # Hinge loss margins
            margins = y * scores
            active = margins < 1

            # Gradients of:
            # 0.5 * ||w||^2 + C * mean(hinge loss)
            dw = self.w - self.C * (
                X[active].T @ y[active]
            ) / n_samples

            db = -self.C * np.sum(y[active]) / n_samples

            # Gradient descent update
            self.w -= self.learning_rate * dw
            self.b -= self.learning_rate * db

            # Track objective value
            scores = X @ self.w + self.b
            hinge = np.maximum(0, 1 - y * scores)
            loss = 0.5 * np.dot(self.w, self.w) + (
                self.C * np.mean(hinge)
            )
            self.loss_history.append(loss)

        return self

    def decision_function(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b

    def predict(self, X):
        scores = self.decision_function(X)
        return np.where(scores >= 0, 1, -1)


# -----------------------------------
# Test the scratch SVM on toy data
# -----------------------------------
if __name__ == "__main__":
    rng = np.random.default_rng(42)

    # Two separable groups
    X_negative = rng.normal(
        loc=[-2, -2], scale=0.8, size=(100, 2)
    )
    X_positive = rng.normal(
        loc=[2, 2], scale=0.8, size=(100, 2)
    )

    X = np.vstack([X_negative, X_positive])
    y = np.array([-1] * 100 + [1] * 100)

    # Shuffle observations
    indices = rng.permutation(len(y))
    X, y = X[indices], y[indices]

    # Standardize features using NumPy
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    X = (X - mean) / std

    # Train
    model = LinearSVM(
        learning_rate=0.01,
        C=1.0,
        epochs=2000
    )
    model.fit(X, y)

    # Evaluate
    predictions = model.predict(X)
    accuracy = np.mean(predictions == y)

    print("Weights:", model.w)
    print("Bias:", model.b)
    print(f"Training accuracy: {accuracy:.3f}")
    print(f"Final objective: {model.loss_history[-1]:.4f}")


Weights: [0.48688631 0.49039256]
Bias: -0.0018999999999999998
Training accuracy: 1.000
Final objective: 0.3882


In [11]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,confusion_matrix

data=load_breast_cancer()
x,y=data.data,data.target
x.shape

(569, 30)

In [14]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
pipeline=Pipeline([('scaler',StandardScaler()),('svm',SVC())])
param_grid={
    "svm__kernel": ["linear", "rbf"],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.1]}

search=GridSearchCV(pipeline,param_grid,cv=5,scoring='accuracy',n_jobs=-1)
search.fit(x_train,y_train)

# 5. Evaluate the best model on unseen test data
best_model = search.best_estimator_
y_pred = best_model.predict(x_test)

print("\nBest parameters:", search.best_params_)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))




Best parameters: {'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Test accuracy: 0.9825
Confusion matrix:
[[41  2]
 [ 0 71]]


In [15]:
from sklearn.datasets import load_diabetes
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

data = load_diabetes()
X, y = data.data, data.target

print("Dataset shape:", X.shape)


Dataset shape: (442, 10)


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])

param_grid = {
    "svr__kernel": ["linear", "rbf"],
    "svr__C": [0.1, 1, 10, 100],
    "svr__epsilon": [0.1, 0.5, 1.0],
    "svr__gamma": ["scale", 0.01, 0.1]
}

search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)
search.fit(X_train, y_train)

# 5. Evaluate on unseen test data
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\nBest parameters:", search.best_params_)
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2_score(y_test, y_pred):.4f}")

# Compare actual and predicted values
print("\nActual vs Predicted:")
for actual, predicted in zip(y_test[:10], y_pred[:10]):
    print(f"Actual: {actual:.1f}, Predicted: {predicted:.1f}")



Best parameters: {'svr__C': 1, 'svr__epsilon': 1.0, 'svr__gamma': 'scale', 'svr__kernel': 'linear'}
MAE:  43.34
RMSE: 54.18
R²:   0.4459

Actual vs Predicted:
Actual: 219.0, Predicted: 133.6
Actual: 70.0, Predicted: 177.5
Actual: 202.0, Predicted: 133.8
Actual: 230.0, Predicted: 286.0
Actual: 111.0, Predicted: 117.8
Actual: 84.0, Predicted: 91.8
Actual: 242.0, Predicted: 238.0
Actual: 272.0, Predicted: 178.2
Actual: 94.0, Predicted: 86.5
Actual: 96.0, Predicted: 105.6
